We realized that there aren't margin limits on taking short positions. 

So testing some strategies to take advantage of that:

In [10]:
import numpy as np
import pandas as pd
import sys

sys.path.insert(0, '..')
from backtester import Backtester

In [23]:
df = pd.read_csv('../dev_data_30.csv', parse_dates=['Date'], date_format='%m/%d/%y', thousands=',')

prices_df = df.pivot(index='Ticker', columns='Date', values='Open').sort_index()
prices_df = prices_df.ffill(axis=1).dropna(axis=1)

prices = prices_df.values

In [24]:
len(prices)

30

In [25]:
def backtest(strategy_fn, prices, cash=25_000):
    """Run strategy_fn on prices and return port_values + pnl."""
    actions = strategy_fn(prices)
    bt = Backtester(prices, actions, cash=cash)
    port_values, pnl = bt.eval_actions()
    if port_values is None:
        return None
    return {
        'port_values': np.array(port_values),
        'pnl': pnl,
    }

print('Backtester ready')

Backtester ready


In [20]:
def find_global_max_stock_day(prices):
    flat_idx = np.argmax(prices)
    stock, day = np.unravel_index(flat_idx, prices.shape)
    price = prices[stock, day]
    return stock, day, price

In [21]:
def get_actions(prices):
    actions = np.zeros_like(prices, dtype=int)

    stock, day, price = find_global_max_stock_day(prices)

    shares_to_short = int(round(1_000_000 / price))
    actions[stock, day] = -shares_to_short

    print(day)

    return actions

In [36]:
initial = 25_000
results = backtest(get_actions, prices, initial)

if results:
    print(f"Initial Portfolio: ${initial:,.2f}")
    print(f"PnL: ${results['pnl']:,.2f}")
else:
    print('Backtest failed — check your strategy for errors.')

987
Final portfolio value: 142980.46
PnL: 117980.46
Initial Portfolio: $25,000.00
PnL: $117,980.46


# Bubble Short
Detect stocks that have run far above their recent average and are starting to revert. 

Short large notional and cover after rebound or to a holding period

In [28]:
def get_actions_bubble_short(prices):
    n_stocks, n_days = prices.shape
    actions = np.zeros_like(prices, dtype=int)

    lookback = 20
    min_history = 25
    entry_notional = 200_000
    max_holding = 20

    # Track whether we currently have an active short from this strategy
    short_age = np.zeros(n_stocks, dtype=int)
    in_short = np.zeros(n_stocks, dtype=bool)

    for t in range(1, n_days):
        for s in range(n_stocks):
            if t < min_history:
                continue

            p_t = prices[s, t]
            p_y = prices[s, t - 1]
            window = prices[s, t - lookback:t]
            ma = np.mean(window)

            # Overextension score
            stretch = p_t / ma if ma > 0 else 1.0

            # Entry: price is very stretched vs recent average, but now weakening
            # Uses only info through day t
            entry_signal = (
                (stretch > 1.25) and      # very extended
                (p_t < p_y) and           # down today vs yesterday
                (p_y >= np.max(window))   # yesterday was at/near recent high
            )

            if not in_short[s] and entry_signal:
                shares = int(round(entry_notional / p_t))
                if shares > 0:
                    actions[s, t] = -shares
                    in_short[s] = True
                    short_age[s] = 0
                continue

            if in_short[s]:
                short_age[s] += 1

                # Exit if mean reversion happened or holding period exceeded
                exit_signal = (
                    (p_t <= ma) or
                    (short_age[s] >= max_holding) or
                    (p_t > 1.05 * p_y)   # squeeze / rebound protection
                )

                if exit_signal:
                    # positive action covers outstanding short
                    # use a huge positive so the engine fully covers
                    actions[s, t] = 10**9
                    in_short[s] = False
                    short_age[s] = 0

    return actions

In [29]:
results = backtest(get_actions_bubble_short, prices)

if results:
    print(f"PnL: ${results['pnl']:,.2f}")
else:
    print('Backtest failed — check your strategy for errors.')

Final portfolio value: 50083.17
PnL: 25083.17
PnL: $25,083.17


# Path Aware
Wait for a stock to make a recent high. Then short only after reversal confirmation. Exit on stop-loss, profit target, or timeout.

In [81]:
import numpy as np

def get_actions_path_aware_exploit(prices):
    n_stocks, n_days = prices.shape
    actions = np.zeros_like(prices, dtype=int)

    lookback = 20
    min_history = 25
    entry_notional = 200_000

    max_holding_days = 20
    stop_loss_pct = 0.08
    profit_take_pct = 0.12

    # Strategy-side bookkeeping
    in_short = np.zeros(n_stocks, dtype=bool)
    entry_price = np.zeros(n_stocks, dtype=float)
    shares_held = np.zeros(n_stocks, dtype=int)
    days_held = np.zeros(n_stocks, dtype=int)

    for t in range(n_days):
        for s in range(n_stocks):
            p_t = prices[s, t]

            # 1. Manage open short
            if in_short[s]:
                days_held[s] += 1

                stop_hit = p_t >= entry_price[s] * (1.0 + stop_loss_pct)
                target_hit = p_t <= entry_price[s] * (1.0 - profit_take_pct)
                timeout_hit = days_held[s] >= max_holding_days

                if stop_hit or target_hit or timeout_hit:
                    actions[s, t] = shares_held[s]
                    in_short[s] = False
                    entry_price[s] = 0.0
                    shares_held[s] = 0
                    days_held[s] = 0

                continue

            # 2. Need enough history before considering entry
            if t < min_history:
                continue

            # Use only data available through day t
            p_y = prices[s, t - 1]
            p_2 = prices[s, t - 2]

            recent_window_through_yesterday = prices[s, t - lookback:t]
            recent_high = np.max(recent_window_through_yesterday)

            # Entry logic:
            # yesterday was at/near a 20d high,
            # and today confirms reversal
            was_near_high = p_y >= recent_high
            down_today = p_t < p_y
            momentum_turned = p_y > p_2 and p_t < p_y

            if was_near_high and down_today and momentum_turned:
                shares = int(round(entry_notional / p_t))
                if shares <= 0:
                    continue

                actions[s, t] = -shares
                in_short[s] = True
                entry_price[s] = p_t
                shares_held[s] = shares
                days_held[s] = 0

    return actions

In [82]:
results = backtest(get_actions_path_aware_exploit, prices)

if results:
    print(f"PnL: ${results['pnl']:,.2f}")
else:
    print('Backtest failed — check your strategy for errors.')

BACKTEST FAILED: portfolio went negative on day 37 (value: -20608.47). Too many short positions.
Backtest failed — check your strategy for errors.
